# ir_calendar_parse_documents — Volume 財報檔案 → `ai_parse_document` → bronze / silver

- 用途：把 `ir_calendar_consume_batches` 落到 Volume 的財報檔案（PDF / HTM），依**公司分類 × 公司**逐檔用 Databricks `ai_parse_document` 做初步解析：原始結果進 bronze、整理成可查閱的全文與元素表進 silver，供後續進階處理（`ai_query` 摘要、表格抽取、RAG 切塊）或人工查閱。取代參考腳本 `job_25_all_parse_content_new.py` 的「一種文件類型一張表 + meta 表 is_parsed 旗標」作法。
- 輸入：`s_{domain}_document_file`（Volume 檔案索引，由 consume_batches 維護：`volume_path`、`sha256`、`category`、`company_key`、`doc_kind`…）與 Volume 上的實體檔案。**不掃目錄、不拆檔名**，分類 / 公司全部取索引欄位。
- 輸出：
  - bronze `b_{domain}_document_parse`：一檔一版（`volume_path, sha256`）一列，`payload` = `ai_parse_document` 輸出 JSON 全文，另存 status / 頁數 / 元素數
  - silver `s_{domain}_document_text`：一檔一列，全文 markdown（`text_md`）
  - silver `s_{domain}_document_element`：一元素（段落 / 表格 / 圖表描述）一列，供切塊與抽取
- 分類與公司是**欄位**不是表：`category` 新增值（未來的新分類）自動進來，不用改表、不用改 code。
- 表由 `ir_calendar_init_tables` 建（[c16]～[c18]）；分層設計見 `docs/20260921_ir_calendar_lakehouse_design.md` 第 8 節。
- 參數（widgets）：見 [c01] 與下表。
- 排程：跟在 `ir_calendar_consume_batches` 之後（同一個 job 的下一個 task，或每天一次）；job cluster。`ai_parse_document` 按頁計費，靠 `max_files` 當費用閘門。
- 負責人 / 更新日期：（填）/ 2026-09-22

## 一次執行做什麼

1. **工作清單** [c05] `select_work`：`s_document_file` 裡 `(volume_path, sha256)` 在 bronze 沒有 `success` 列的檔案；最新一列是 `failed` / `partial` 且 `attempt < max_attempts` 的會重試。可用 `categories` / `company_keys` / `doc_kinds` 縮小範圍；`force_reparse = true` 時不看 bronze，範圍內全部重解析（換 parser 版本或選項時用）。取前 `max_files` 個，依 分類 → 公司 → 路徑 排序。
2. **分組** [c03] `plan_chunks`：先依解析選項分組（一次 `ai_parse_document` 呼叫只能一組選項；簡報開圖表描述、其餘不開），再切成每 `chunk_size` 個一組。
3. **每組**：`binaryFile` 讀 Volume 檔案（內容留在 executor）→ `ai_parse_document` → `CAST AS STRING` → [c05] `transform_parse_result` 判 status、算頁數 / 元素數 / 字元數 → 先 `DELETE` 同鍵舊列再 append（重跑冪等）。
   失敗（rate limit、逾時）依 [c03] `backoff_seconds` 重試 `max_retries` 次；仍失敗就寫一列 `status = failed`（無 payload、`attempt + 1`），**繼續下一組**。組間等 `pause_seconds`。Volume 上找不到的檔案直接記 failed，不呼叫。
4. **silver**：本次寫入的 bronze 列 → [c06] transform → `document_text` MERGE、`document_element` 以文件為單位刪後 append（元素數會變，MERGE 清不掉多出來的舊元素）。
5. 印統計；有 failed 且 `fail_on_error = true` 就 `raise`，讓 Databricks job 顯示失敗。

**重跑安全**：bronze 以 `(volume_path, sha256)` 刪後 append；silver 同鍵覆蓋。同一路徑的新版本檔（sha256 不同）是新的一列，舊版保留。中途失敗的組下次自動重排（attempt 未達上限）。

**重算 silver**：`rebuild_silver = true` 時不解析，只從整張 bronze 重算兩張 silver（同鍵取最新 `parsed_at`；`document_element` 整表 overwrite）。

## 參數

| widget | 預設 | 說明 |
|---|---|---|
| `catalog` / `schema` | `micenter` / `mi3_datahub_prod` | Delta table 所在；對照 `config/project.yml` |
| `domain` | `ir_calendar` | 表名中段；層級前綴 `b_` / `s_` 固定 |
| `categories` | 空 | 逗號分隔的分類代碼（`CUSTOMER,SUPPLIER`）；空 = 全部 |
| `company_keys` | 空 | 逗號分隔的 `company_key`（`HK:1070,TPE:2353`）；空 = 全部 |
| `doc_kinds` | 空 | 逗號分隔的 `doc_kind`（`Presentation,Financial_Statements`）；空 = 全部 |
| `max_files` | `50` | 單次最多解析幾個檔案（費用與時間上限）；首次追趕分幾次跑或調大 |
| `chunk_size` | `1` | 一次交給 `ai_parse_document` 幾個檔案。`1` = 逐檔最穩、失敗只影響一檔；確認不會撞 rate limit 再調大 |
| `max_retries` | `3` | 一組失敗的重試次數 |
| `pause_seconds` | `15` | 組與組之間的等待秒數（參考腳本經驗值，避免連續呼叫觸發 rate limit） |
| `max_attempts` | `3` | 同一檔累計 `attempt` 達到就不再排入；要再試就 `force_reparse` 搭配篩選 |
| `force_reparse` | `false` | `true`：範圍內全部重解析，不看 bronze。**一定搭配 `categories` / `company_keys` / `doc_kinds` 或小 `max_files`，避免整批重花費** |
| `rebuild_silver` | `false` | `true`：不解析，只從整張 bronze 重算 silver |
| `dry_run` | `false` | `true`：只列出會解析哪些檔案，不呼叫 `ai_parse_document`、不寫表 |
| `fail_on_error` | `true` | `true`：本次有 `failed` 就讓 job 失敗（通知才會發）；壞檔會反覆失敗時可暫時關掉 |
| `job_run_id` | 空 | job parameters 填 `{{job.run_id}}`，寫進 bronze 方便對 log |

## 維護者須知

| 要改什麼 | 改哪裡 |
|---|---|
| 新增公司分類 | **不用改**：`category` 是欄位，值來自 `s_document_file` |
| 新增 `doc_kind` | 不用改；要對它開圖表描述才改 [c02] `DESCRIBE_FIGURE_DOC_KINDS` |
| `ai_parse_document` 版本 / 選項 | [c02] `PARSER_VERSION`、[c03] `parse_options`；輸出形狀變了改 [c04] `PARSE_PAYLOAD` + [c05] `status_col` / `text_md_col` + [c06] transform。bronze `parse_options` 記錄每列當時的選項，改完用 `force_reparse` 分批重解析 |
| 輸出形狀還沒確認 | [c04] `PARSE_PAYLOAD` 是依官方文件寫的**假設**（`document.pages[]`、`document.elements[]{id,type,content,description,page_id}`、`error_status`）。首次跑完先 `SELECT payload:metadata, payload:document.elements[0] FROM b_..._document_parse LIMIT 1` 對一下，不合就改 [c04]，再 `rebuild_silver` |
| silver 要多欄位 | 建表 notebook `ALTER TABLE ADD COLUMNS` → [c04] `PARSE_PAYLOAD` → [c06] transform 的 `select` → `rebuild_silver = true` |
| 費用 | `ai_parse_document` 按頁計費：`max_files` 是閘門；`force_reparse` 一定縮範圍；圖表描述只對簡報開 |
| 只想本機測邏輯 | [c02]～[c06] 沒有 I/O、沒有 `spark` 全域變數 / `dbutils`；`tests/test_ir_calendar_parse_documents.py` 用假的 `ai_parse_document` JSON 跑 transform |

## 首次上線順序

1. 跑 `ir_calendar_init_tables`（已含 [c16]～[c18] 三張新表；既有表不受影響）
2. `dry_run = true`：看列出的檔案、分類 / 公司統計對不對
3. `max_files = 2`、`dry_run = false` 跑一次：對 bronze `payload` 形狀（見維護者須知），看 silver 兩張表內容
4. 調 `max_files` 追完存量；之後排程接在 consume_batches 後面


In [ ]:
# [c01] params
# 預設值取自 config/project.yml（catalog / schema）；job 執行時由 job parameters 覆蓋。
# 注意：widget 一旦在 notebook 建立過，改 code 的預設值不會更新既有 widget；要重設請先 dbutils.widgets.removeAll() 再跑本 cell。布林 / 數字都用字串填，這裡統一轉型。
dbutils.widgets.text("catalog", "micenter")
dbutils.widgets.text("schema", "mi3_datahub_prod")
dbutils.widgets.text("domain", "ir_calendar")     # 表名中段；層級前綴 b_ / s_ 固定，見 docs/conventions.md 2.1
dbutils.widgets.text("categories", "")            # 逗號分隔的分類代碼（CUSTOMER,SUPPLIER…）；空 = 全部
dbutils.widgets.text("company_keys", "")          # 逗號分隔的 company_key（HK:1070,TPE:2353）；空 = 全部
dbutils.widgets.text("doc_kinds", "")             # 逗號分隔的 doc_kind；空 = 全部
dbutils.widgets.text("max_files", "50")           # 單次最多解析幾個檔案（ai_parse_document 按頁計費，這是費用閘門）
dbutils.widgets.text("chunk_size", "1")           # 一次交給 ai_parse_document 幾個檔案；1 = 逐檔最穩，確認不撞 rate limit 再調大
dbutils.widgets.text("max_retries", "3")          # 一組失敗的重試次數
dbutils.widgets.text("pause_seconds", "15")       # 組與組之間的等待秒數，避免連續呼叫觸發 rate limit
dbutils.widgets.text("max_attempts", "3")         # 同一檔累計失敗幾次後不再排入（要再試就 force_reparse 或調高）
dbutils.widgets.text("force_reparse", "false")    # true：範圍內全部重解析，不看 bronze（換 parser 版本 / 選項時用，一定搭配篩選）
dbutils.widgets.text("rebuild_silver", "false")   # true：不解析，只從整張 bronze 重算 silver
dbutils.widgets.text("dry_run", "false")          # true：只列出會解析哪些檔案，不呼叫 ai_parse_document、不寫表
dbutils.widgets.text("fail_on_error", "true")     # true：本次有 failed 的檔案就讓 job 失敗（通知才會發）
dbutils.widgets.text("job_run_id", "")            # job parameters 填 {{job.run_id}}


def _flag(name: str) -> bool:
    return dbutils.widgets.get(name).strip().lower() == "true"


def _csv(name: str) -> list[str]:
    return [x.strip() for x in dbutils.widgets.get(name).split(",") if x.strip()]


settings = {
    "catalog": dbutils.widgets.get("catalog").strip(),
    "schema": dbutils.widgets.get("schema").strip(),
    "domain": dbutils.widgets.get("domain").strip(),
    "categories": _csv("categories"),
    "company_keys": _csv("company_keys"),
    "doc_kinds": _csv("doc_kinds"),
    "max_files": int(dbutils.widgets.get("max_files")),
    "chunk_size": int(dbutils.widgets.get("chunk_size")),
    "max_retries": int(dbutils.widgets.get("max_retries")),
    "pause_seconds": float(dbutils.widgets.get("pause_seconds")),
    "max_attempts": int(dbutils.widgets.get("max_attempts")),
    "force_reparse": _flag("force_reparse"),
    "rebuild_silver": _flag("rebuild_silver"),
    "dry_run": _flag("dry_run"),
    "fail_on_error": _flag("fail_on_error"),
    "job_run_id": dbutils.widgets.get("job_run_id").strip() or None,
}
assert settings["catalog"] and settings["schema"] and settings["domain"], "catalog / schema / domain 不可為空"
assert settings["max_files"] >= 1 and settings["chunk_size"] >= 1, "max_files / chunk_size 至少 1"
assert settings["max_retries"] >= 1 and settings["max_attempts"] >= 1, "max_retries / max_attempts 至少 1"
if settings["force_reparse"] and not (settings["categories"] or settings["company_keys"] or settings["doc_kinds"]):
    print("!!! force_reparse 沒有搭配任何篩選：範圍內最多 max_files 個檔案會重新計費解析")
print(settings)


In [ ]:
# [c02] imports
# 共用 import 與常數。本 cell 與 [c03]～[c06] 不碰 spark 全域變數 / dbutils / 檔案系統，可被 tests/ 載入。
import json
import os
import time
from collections import Counter
from datetime import datetime, timezone

from pyspark.sql import Column, DataFrame, Window
from pyspark.sql import functions as F

UTC = timezone.utc  # noqa: UP017 - 本機測試仍是 Python 3.10，不用 datetime.UTC

# 表命名：<層級前綴><domain>_<短名>，層級前綴固定（docs/conventions.md 2.1）
LAYER_PREFIX = {"bronze": "b_", "silver": "s_"}
SOURCE_TABLE = "document_file"      # 工作清單來源：s_{domain}_document_file（consume_batches 維護，本 job 不改它）
PARSE_TABLE = "document_parse"      # bronze：b_{domain}_document_parse（本 job 的原始解析結果，含 payload 原文）

# ai_parse_document 的版本。版本換了：改這裡 + [c04] PARSE_PAYLOAD + [c05] status_col / [c03] text_md_col + [c06] transform。
PARSER = "ai_parse_document"
PARSER_VERSION = "2.0"
# 這些 doc_kind 開圖表描述（descriptionElementTypes=figure）：簡報的圖表沒有文字，要描述才查得到；
# 財報 / 新聞稿 / 逐字稿以文字為主，不開省費用。doc_kind 值來自爬蟲 manifest（Presentation / Financial_Statements / Press_Release / Earnings_Transcript）。
DESCRIBE_FIGURE_DOC_KINDS = {"Presentation"}

# bronze.status：success = 整份解析成功 / partial = 部分頁失敗（payload.error_status 非空）/ failed = 呼叫失敗、找不到檔或無結果
STATUS_SUCCESS, STATUS_PARTIAL, STATUS_FAILED = "success", "partial", "failed"
RETRY_STATUSES = (STATUS_PARTIAL, STATUS_FAILED)
RATE_LIMIT_HINTS = ("429", "rate limit", "resource_exhausted", "quota", "too many requests")


In [ ]:
# [c03] pure_helpers
# 純函式：解析選項、分組、重試等待、欄位表達式。無 I/O。


def parse_options(doc_kind: str | None) -> dict[str, str]:
    """一個檔案要給 ai_parse_document 的選項 map。目前只有版本與「要不要描述圖表」兩件事，依 doc_kind 決定。"""
    opts = {"version": PARSER_VERSION}
    if doc_kind in DESCRIBE_FIGURE_DOC_KINDS:
        opts["descriptionElementTypes"] = "figure"
    return opts


def options_key(opts: dict[str, str]) -> str:
    """選項的穩定字串（鍵排序）：同一組 chunk 的檔案選項必須相同；也原樣存進 bronze.parse_options。"""
    return json.dumps(opts, sort_keys=True, ensure_ascii=False)


def _q(s: str) -> str:
    return "'" + s.replace("'", "''") + "'"


def options_sql(opts: dict[str, str]) -> str:
    """選項 map → SQL 的 map('k', 'v', …) 字面值。鍵值只會是我們自己的常數，仍做單引號跳脫。"""
    parts = [x for k, v in sorted(opts.items()) for x in (_q(k), _q(v))]
    return "map(" + ", ".join(parts) + ")"


def to_work(row: dict) -> dict:
    """select_work 的一列 → 工作項：補上 options_key。"""
    return {**row, "options_key": options_key(parse_options(row.get("doc_kind")))}


def plan_chunks(work: list[dict], chunk_size: int) -> list[list[dict]]:
    """工作清單 → 每組最多 chunk_size 個檔案。先依 options_key 分組（一次 ai_parse_document 呼叫只能一組選項），
    組內依 分類 → 公司 → 路徑 排序，log 讀起來是一家公司一家公司處理。"""
    groups: dict[str, list[dict]] = {}
    ordered = sorted(work, key=lambda w: (w.get("category") or "", w.get("company_key") or "", w["volume_path"]))
    for w in ordered:
        groups.setdefault(w["options_key"], []).append(w)
    chunks: list[list[dict]] = []
    for key in sorted(groups):
        rows = groups[key]
        chunks += [rows[i:i + chunk_size] for i in range(0, len(rows), chunk_size)]
    return chunks


def is_rate_limit(err: BaseException) -> bool:
    s = str(err).lower()
    return any(h in s for h in RATE_LIMIT_HINTS)


def backoff_seconds(attempt: int, rate_limited: bool) -> int:
    """第 attempt 次失敗後等多久：rate limit 120s × attempt，其他 30s × attempt（沿用參考腳本的經驗值）。"""
    return (120 if rate_limited else 30) * attempt


def strip_dbfs(c: Column) -> Column:
    """binaryFile 讀出的 path 是 dbfs:/Volumes/…，去掉 dbfs: 前綴對回 volume_path。"""
    return F.regexp_replace(c, r"^dbfs:", "")


def _nz(c: Column) -> Column:
    return F.nullif(F.trim(c), F.lit(""))


def text_md_col(elements: Column) -> Column:
    """元素陣列（PARSE_PAYLOAD.document.elements）→ 全文 markdown。
    依陣列順序（= 文件順序）串接；content 空的（圖表）用 description 代替；兩者皆空的略過。陣列為 NULL 回 NULL。"""
    piece = F.transform(elements, lambda e: F.coalesce(_nz(e["content"]), _nz(e["description"])))
    joined = F.concat_ws("\n\n", F.filter(piece, lambda x: x.isNotNull()))
    return F.when(elements.isNull(), F.lit(None).cast("string")).otherwise(joined)


def size_or_null(c: Column) -> Column:
    """size()：ANSI 開關不同時 NULL 陣列會回 -1 或 NULL，這裡統一回 NULL。"""
    return F.when(c.isNull(), F.lit(None).cast("int")).otherwise(F.size(c))


In [ ]:
# [c04] table_schemas
#   WORK_SCHEMA  ：工作清單（select_work 的輸出 + options_key），createDataFrame 用
#   PARSE_SCHEMA ：bronze b_*_document_parse，欄位名必須與建表 notebook [c16] 的 DDL 一致
#   PARSE_PAYLOAD：from_json 解析 ai_parse_document 輸出用，只列 silver 會用到的欄位（多的忽略、少的補 NULL）
#   TABLE_KEYS   ：silver 的鍵
from pyspark.sql.types import (
    ArrayType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)


def _s(name: str, dtype=None, nullable: bool = True) -> StructField:
    return StructField(name, dtype or StringType(), nullable)


# 文件識別 + 從 s_document_file 抄過來的屬性：bronze / silver 都帶，不用 join 就能依分類 / 公司篩
DOC_KEY = ["volume_path", "sha256"]
DOC_ATTRS = ["file_name", "company_key", "company_slug", "category", "period", "doc_kind", "fiscal_label", "bytes"]
_DOC_FIELDS = [
    _s("volume_path", nullable=False), _s("sha256"), _s("file_name"), _s("company_key"), _s("company_slug"),
    _s("category"), _s("period"), _s("doc_kind"), _s("fiscal_label"), _s("bytes", LongType()),
]

WORK_SCHEMA = StructType([*_DOC_FIELDS, _s("attempt", IntegerType()), _s("options_key")])
WORK_FIELDS = [f.name for f in WORK_SCHEMA.fields]

PARSE_SCHEMA = StructType([
    *_DOC_FIELDS,
    _s("parser"), _s("parser_version"), _s("parse_options"), _s("attempt", IntegerType(), False),
    _s("status", nullable=False), _s("error"),
    _s("page_count", IntegerType()), _s("element_count", IntegerType()), _s("text_chars", LongType()),
    _s("payload"), _s("parsed_at", TimestampType(), False), _s("job_run_id"),
])

TABLE_KEYS: dict[str, list[str]] = {
    "document_text": DOC_KEY,
    "document_element": [*DOC_KEY, "element_id"],
}

# ---- ai_parse_document 2.0 輸出的形狀（依官方文件；只列用到的，bbox 等不列，要看去 payload）----
# 假設：{"document": {"pages": [{"id"}], "elements": [{"id","type","content","description","page_id"}]},
#        "metadata": {"version","id"}, "error_status": [...] 或 null}
# 首次跑完請對一次 payload，不合就改這裡 + [c03] text_md_col / [c05] status_col / [c06]，再 rebuild_silver。
ELEMENT_STRUCT = StructType([
    _s("id", IntegerType()), _s("type"), _s("content"), _s("description"), _s("page_id", IntegerType()),
])
PARSE_PAYLOAD = StructType([
    _s("document", StructType([
        _s("pages", ArrayType(StructType([_s("id", IntegerType())]))),
        _s("elements", ArrayType(ELEMENT_STRUCT)),
    ])),
    _s("metadata", StructType([_s("version"), _s("id")])),
])


In [ ]:
# [c05] transform_parse
# 純函式 DataFrame -> DataFrame：
#   select_work           ：s_document_file × b_document_parse → 待解析清單（WORK_SCHEMA 去掉 options_key）
#   transform_parse_result：binaryFile + ai_parse_document 的結果（WORK_FIELDS + payload 字串）→ bronze 列（PARSE_SCHEMA 形狀）
#   failed_rows           ：整組呼叫失敗 / 找不到檔時的 bronze 列（Python dict）


def latest_per_doc(df: DataFrame) -> DataFrame:
    """同一 (volume_path, sha256) 多列時只留 parsed_at 最大的。"""
    w = Window.partitionBy(*DOC_KEY).orderBy(F.col("parsed_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")


def select_work(df_docs: DataFrame, df_parse: DataFrame, *, categories=(), company_keys=(), doc_kinds=(),
                max_attempts: int = 3, force: bool = False) -> DataFrame:
    """待解析清單。
    - bronze 沒有這個 (volume_path, sha256) → 排入（attempt = 0）
    - bronze 最新一列是 failed / partial 且 attempt < max_attempts → 重試（attempt 沿用，寫入時 +1）
    - 最新一列 success → 跳過
    - force = True → 不看 status，範圍內全部排入（attempt 沿用）
    篩選條件空 = 不篩。輸出欄位 = DOC_KEY + DOC_ATTRS + attempt。"""
    d = df_docs.select(*DOC_KEY, *DOC_ATTRS)
    if categories:
        d = d.filter(F.col("category").isin(list(categories)))
    if company_keys:
        d = d.filter(F.col("company_key").isin(list(company_keys)))
    if doc_kinds:
        d = d.filter(F.col("doc_kind").isin(list(doc_kinds)))
    latest = latest_per_doc(df_parse.select(*DOC_KEY, "status", "attempt", "parsed_at")).drop("parsed_at")
    j = d.join(latest, DOC_KEY, "left")
    attempt = F.coalesce(F.col("attempt"), F.lit(0)).cast("int")
    if not force:
        retry = F.col("status").isin(list(RETRY_STATUSES)) & (attempt < F.lit(max_attempts))
        j = j.filter(F.col("status").isNull() | retry)
    return j.select(*DOC_KEY, *DOC_ATTRS, attempt.alias("attempt"))


def status_col(payload: Column, parsed: Column) -> Column:
    """payload 原文 + from_json 結果 → status。
    沒有 payload 或 document 解不出來 → failed；error_status 有內容 → partial；否則 success。
    error_status 用 get_json_object 看原文，輸出形狀改了（陣列 / 物件 / 字串）也不會漏判。"""
    err = F.get_json_object(payload, "$.error_status")
    has_err = err.isNotNull() & ~err.isin("null", "[]", "{}", "")
    return (F.when(payload.isNull() | parsed["document"].isNull(), F.lit(STATUS_FAILED))
            .when(has_err, F.lit(STATUS_PARTIAL))
            .otherwise(F.lit(STATUS_SUCCESS)))


def transform_parse_result(df: DataFrame, *, now: datetime, job_run_id: str | None) -> DataFrame:
    """輸入：WORK_FIELDS + payload（ai_parse_document 輸出 CAST AS STRING）。輸出：PARSE_SCHEMA 形狀。
    payload 不動、原樣進 bronze；這裡只多算 status / 頁數 / 元素數 / 字元數，方便不展開 JSON 就能查。"""
    d = df.withColumn("p", F.from_json(F.col("payload"), PARSE_PAYLOAD))
    p, payload = F.col("p"), F.col("payload")
    els = F.col("p.document.elements")
    status = status_col(payload, p)
    error = (F.when(status == STATUS_FAILED, F.lit("no document in ai_parse_document output"))
             .when(status == STATUS_PARTIAL, F.substring(F.get_json_object(payload, "$.error_status"), 1, 2000)))
    return d.select(
        *[F.col(c) for c in DOC_KEY + DOC_ATTRS],
        F.lit(PARSER).alias("parser"), F.lit(PARSER_VERSION).alias("parser_version"),
        F.col("options_key").alias("parse_options"),
        (F.coalesce(F.col("attempt"), F.lit(0)) + 1).cast("int").alias("attempt"),
        status.alias("status"), error.alias("error"),
        size_or_null(F.col("p.document.pages")).alias("page_count"),
        size_or_null(els).alias("element_count"),
        F.length(text_md_col(els)).cast("long").alias("text_chars"),
        payload,
        F.lit(now).cast("timestamp").alias("parsed_at"),
        F.lit(job_run_id).cast("string").alias("job_run_id"),
    )


def failed_rows(chunk: list[dict], error: str, *, now: datetime, job_run_id: str | None) -> list[dict]:
    """整組呼叫失敗（重試用盡）或檔案不存在時的 bronze 列：status = failed、沒有 payload、attempt + 1。"""
    return [{
        **{k: w.get(k) for k in DOC_KEY + DOC_ATTRS},
        "parser": PARSER, "parser_version": PARSER_VERSION, "parse_options": w["options_key"],
        "attempt": int(w.get("attempt") or 0) + 1, "status": STATUS_FAILED, "error": str(error)[:2000],
        "page_count": None, "element_count": None, "text_chars": None, "payload": None,
        "parsed_at": now, "job_run_id": job_run_id,
    } for w in chunk]


In [ ]:
# [c06] transform_silver
# bronze b_*_document_parse → silver 的純函式 DataFrame -> DataFrame。輸入是 PARSE_SCHEMA 形狀，輸出是 silver 表欄位
# （不含 updated_at，由 [c08] 寫入時補）。只拿有 payload 的列（success / partial），同鍵取最新 parsed_at。
# 加 silver 表：寫 transform_<短名>，登記到 SILVER_TRANSFORMS，鍵放 [c04] TABLE_KEYS，DDL 放建表 notebook。


LINEAGE_SRC = ["parser", "parser_version", "parse_options", "status", "parsed_at"]   # bronze 上的血緣欄位


def _lineage() -> list[Column]:
    """bronze 血緣：用哪個 parser / 哪組選項 / 何時解析、status 是 success 還是 partial（改名 parse_status）。"""
    return [F.col("parser"), F.col("parser_version"), F.col("parse_options"),
            F.col("status").alias("parse_status"), F.col("parsed_at")]


def _parsed(df: DataFrame) -> DataFrame:
    d = latest_per_doc(df.filter(F.col("payload").isNotNull()))
    return d.withColumn("p", F.from_json(F.col("payload"), PARSE_PAYLOAD))


def transform_document_text(df: DataFrame) -> DataFrame:
    """一檔一列：全文 markdown（元素依序串接，圖表以描述代替）。給 ai_query 摘要、全文檢索、人工查閱。"""
    d = _parsed(df)
    return d.select(
        *DOC_KEY, *DOC_ATTRS,
        F.col("page_count"), F.col("element_count"),
        text_md_col(F.col("p.document.elements")).alias("text_md"),
        F.col("text_chars"),
        *_lineage(),
    )


def transform_document_element(df: DataFrame) -> DataFrame:
    """一元素一列：段落 / 標題 / 表格 / 圖表（描述）。seq 是在文件內的順序；element_id 取 ai_parse_document 給的 id，沒有就用 seq。"""
    d = _parsed(df)
    e = F.col("e")
    return d.select(
        *DOC_KEY, *DOC_ATTRS, F.posexplode(F.col("p.document.elements")).alias("seq", "e"), *LINEAGE_SRC,
    ).select(
        *DOC_KEY, *DOC_ATTRS,
        F.coalesce(e["id"], F.col("seq")).cast("int").alias("element_id"),
        F.col("seq").cast("int").alias("seq"),
        e["page_id"].alias("page_id"), e["type"].alias("element_type"),
        e["content"].alias("content"), e["description"].alias("description"),
        *_lineage(),
    )


# 短名 → transform。順序 = 寫入順序。
SILVER_TRANSFORMS = {
    "document_text": transform_document_text,
    "document_element": transform_document_element,
}


In [ ]:
# [c07] io_parse
# 真的呼叫 ai_parse_document：binaryFile 讀 Volume 檔案（內容留在 executor，不進 driver）→ 一組一次 SQL 呼叫 → CAST AS STRING。
# 這裡只組 lazy DataFrame；執行、重試、寫入在 [c08] write_parse_chunk。
# ai_parse_document 是 Databricks AI Functions（需 workspace 啟用、區域支援）；VARIANT 輸出 CAST 成 STRING 進 bronze，
# 理由同 b_record.payload（docs/20260921_ir_calendar_lakehouse_design.md 第 5 節）。


def read_chunk_binary(chunk: list[dict]) -> DataFrame:
    paths = [w["volume_path"] for w in chunk]
    return (spark.read.format("binaryFile").load(paths)
            .select(strip_dbfs(F.col("path")).alias("volume_path"), F.col("content")))


def parse_chunk(chunk: list[dict]) -> DataFrame:
    """chunk 內所有檔案的選項相同（plan_chunks 保證）。回 WORK_FIELDS + payload（STRING JSON）。"""
    opts = json.loads(chunk[0]["options_key"])
    df_work = spark.createDataFrame([{k: w.get(k) for k in WORK_FIELDS} for w in chunk], schema=WORK_SCHEMA)
    call = f"CAST({PARSER}(content, {options_sql(opts)}) AS STRING)"
    return (read_chunk_binary(chunk).join(F.broadcast(df_work), "volume_path")
            .select(*[F.col(c) for c in WORK_FIELDS], F.expr(call).alias("payload")))


In [ ]:
# [c08] io_delta
#   write_parse_chunk：一組：跑 [c07] → [c05] transform → DELETE 同 (volume_path, sha256) → append。失敗依 backoff 重試，
#                      用盡就寫 failed 列。回錯誤字串（None = 成功）
#   write_failed     ：找不到檔等不必呼叫的情況，直接寫 failed 列
#   write_silver     ：拿 bronze 列跑 [c06] transform → document_text MERGE；document_element 以文件為單位 DELETE 再 append
#                      （元素數會隨重新解析改變，MERGE 清不掉多出來的舊元素，整份文件替換語意等同覆蓋）。full = True 時整表 overwrite
# dry_run 時不會走到這裡。表不存在會直接拋錯：先跑 ir_calendar_init_tables。
from delta.tables import DeltaTable


def table_name(settings: dict, layer: str, short: str) -> str:
    return f"{settings['catalog']}.{settings['schema']}.{LAYER_PREFIX[layer]}{settings['domain']}_{short}"


def _doc_keys_df(rows: list[dict]) -> DataFrame:
    return spark.createDataFrame([(r["volume_path"], r.get("sha256")) for r in rows],
                                 schema="volume_path string, sha256 string")


def _delete_docs(table: str, df_keys: DataFrame) -> None:
    """依 (volume_path, sha256) 刪列。用 MERGE … whenMatchedDelete，鍵多也不必拼超長 WHERE。"""
    (DeltaTable.forName(spark, table).alias("t")
     .merge(df_keys.select(*DOC_KEY).distinct().alias("s"),
            "t.volume_path = s.volume_path AND t.sha256 <=> s.sha256")
     .whenMatchedDelete()
     .execute())


def _merge(table: str, df: DataFrame, keys: list[str]) -> None:
    cond = " AND ".join(f"t.{k} <=> s.{k}" for k in keys)
    (DeltaTable.forName(spark, table).alias("t")
     .merge(df.alias("s"), cond)
     .whenMatchedUpdateAll()
     .whenNotMatchedInsertAll()
     .execute())


def write_failed(settings: dict, chunk: list[dict], error: str, now: datetime) -> None:
    table = table_name(settings, "bronze", PARSE_TABLE)
    rows = failed_rows(chunk, error, now=now, job_run_id=settings["job_run_id"])
    _delete_docs(table, _doc_keys_df(rows))
    spark.createDataFrame(rows, schema=PARSE_SCHEMA).write.format("delta").mode("append").saveAsTable(table)


def write_parse_chunk(settings: dict, chunk: list[dict], now: datetime) -> str | None:
    table = table_name(settings, "bronze", PARSE_TABLE)
    last_error = ""
    for attempt in range(1, settings["max_retries"] + 1):
        try:
            df_out = transform_parse_result(parse_chunk(chunk), now=now, job_run_id=settings["job_run_id"])
            _delete_docs(table, _doc_keys_df(chunk))      # 重跑 / 重試：先清同鍵舊列；append 失敗不會留半套（Delta 交易）
            df_out.write.format("delta").mode("append").saveAsTable(table)
            return None
        except Exception as e:  # noqa: BLE001 - 任何錯都重試，最後寫 failed 列，不讓一組壞檔拖垮整批
            last_error = repr(e)
            if attempt >= settings["max_retries"]:
                break
            rate_limited = is_rate_limit(e)
            wait = backoff_seconds(attempt, rate_limited)
            print(f"    第 {attempt} 次失敗，{wait}s 後重試{'（rate limit）' if rate_limited else ''}：{last_error[:200]}")
            time.sleep(wait)
    print(f"    重試 {settings['max_retries']} 次仍失敗，寫 failed 列：{last_error[:200]}")
    write_failed(settings, chunk, last_error, now)
    return last_error


def write_silver(settings: dict, df_bronze: DataFrame, *, full: bool = False) -> dict[str, int]:
    """df_bronze 可以是本次寫入的列（增量）或整張表（rebuild）。回各表寫入列數。"""
    out = {}
    for short, transform in SILVER_TRANSFORMS.items():
        table = table_name(settings, "silver", short)
        df = transform(df_bronze).withColumn("updated_at", F.current_timestamp())
        if full:
            df.write.format("delta").mode("overwrite").saveAsTable(table)       # 整表重算：直接覆蓋，schema 不變
            out[short] = spark.table(table).count()
        elif short == "document_element":
            _delete_docs(table, df_bronze.select(*DOC_KEY))
            df.write.format("delta").mode("append").saveAsTable(table)
            out[short] = df.count()
        else:
            _merge(table, df, TABLE_KEYS[short])
            out[short] = df.count()
        print(f"    {table}: {'overwrite' if full else 'write'} {out[short]} 列")
    return out


In [ ]:
# [c10] run_once
# 一次執行：工作清單 → 分組 → 逐組解析寫 bronze → 本次寫入的列算 silver。


def run_once(settings: dict) -> dict:
    run_start = datetime.now(UTC)
    df_docs = spark.table(table_name(settings, "silver", SOURCE_TABLE))
    df_parse = spark.table(table_name(settings, "bronze", PARSE_TABLE))
    df_work = select_work(
        df_docs, df_parse, categories=settings["categories"], company_keys=settings["company_keys"],
        doc_kinds=settings["doc_kinds"], max_attempts=settings["max_attempts"], force=settings["force_reparse"],
    ).orderBy("category", "company_key", "volume_path")
    work = [to_work(r.asDict()) for r in df_work.limit(settings["max_files"]).collect()]   # 最多 max_files 列，小
    report = {"work": len(work), "chunks": 0, "counts": {}, "errors": [], "silver": {}}
    print(f"待解析 {len(work)} 個檔案（max_files={settings['max_files']}，篩選 categories={settings['categories']} "
          f"company_keys={settings['company_keys']} doc_kinds={settings['doc_kinds']} force={settings['force_reparse']}）")
    by_company = Counter((w["category"] or "", w["company_key"] or "", w["company_slug"] or "") for w in work)
    for (cat, ck, slug), n in sorted(by_company.items()):
        print(f"  {cat:<12} {ck:<12} {slug:<30} {n} 檔")
    if not work:
        return report
    if settings["dry_run"]:
        for w in work:
            print(f"  [dry_run] attempt={w['attempt']} {w['doc_kind']:<22} {w['volume_path']}")
        return report

    chunks = plan_chunks(work, settings["chunk_size"])
    report["chunks"] = len(chunks)
    for i, chunk in enumerate(chunks, 1):
        now = datetime.now(UTC)
        head = chunk[0]
        print(f"[{i}/{len(chunks)}] {head['category']}/{head['company_slug']} {len(chunk)} 檔 opts={head['options_key']}")
        for w in chunk:
            print(f"    {w['doc_kind']:<22} {w['file_name']}")
        missing = [w for w in chunk if not os.path.exists(w["volume_path"])]     # Volume 走 FUSE 路徑，os 可讀
        if missing:
            write_failed(settings, missing, "file not found on Volume", now)
            report["errors"] += [f"{w['volume_path']}: file not found" for w in missing]
            chunk = [w for w in chunk if w not in missing]
        if chunk:
            err = write_parse_chunk(settings, chunk, now)
            if err:
                report["errors"] += [f"{w['volume_path']}: {err[:200]}" for w in chunk]
        if i < len(chunks) and settings["pause_seconds"] > 0:
            time.sleep(settings["pause_seconds"])

    # 本次寫入的 bronze 列：印每檔結果、統計、算 silver（≤ max_files 列，collect 可接受）
    df_new = spark.table(table_name(settings, "bronze", PARSE_TABLE)).filter(F.col("parsed_at") >= F.lit(run_start))
    rows = df_new.select("category", "company_slug", "doc_kind", "file_name", "status", "attempt", "page_count",
                         "element_count", "text_chars", "error").orderBy("category", "company_slug", "file_name").collect()
    print("\n本次結果：")
    for r in rows:
        extra = (f"pages={r.page_count} elements={r.element_count} chars={r.text_chars}"
                 if r.status != STATUS_FAILED else (r.error or "")[:120])
        print(f"  {r.status:<8} a{r.attempt} {r.category:<12} {r.company_slug:<24} {r.doc_kind:<22} {r.file_name}  {extra}")
    report["counts"] = dict(Counter(r.status for r in rows))
    print(f"統計：{report['counts']}")
    report["silver"] = write_silver(settings, df_new, full=False)
    return report


In [ ]:
# [c11] rebuild_silver
# 從整張 bronze 重算 silver：改 PARSE_PAYLOAD / transform、或懷疑 silver 壞掉時用（rebuild_silver = true）。
# 不呼叫 ai_parse_document、不動 bronze。同鍵取最新 parsed_at（latest_per_doc），所以重算結果 = 逐次增量的結果。


def rebuild_silver(settings: dict) -> dict[str, int]:
    df_all = spark.table(table_name(settings, "bronze", PARSE_TABLE))
    print(f"rebuild silver from {table_name(settings, 'bronze', PARSE_TABLE)}")
    return write_silver(settings, df_all, full=True)


In [ ]:
# [c20] main
# job 進入點。rebuild_silver = true 只重算 silver；否則解析新檔案。有 failed 且 fail_on_error 就 raise，讓 Databricks job 顯示失敗、通知才會發。
if settings["rebuild_silver"]:
    print("silver 重算結果：", rebuild_silver(settings))
else:
    report = run_once(settings)
    if report["errors"] and settings["fail_on_error"]:
        raise RuntimeError(f"{len(report['errors'])} 個檔案解析失敗：" + "; ".join(report["errors"][:10]))


In [ ]:
# [c30] check_tables
# 收尾：bronze 每檔最新一列依分類 × status 統計、silver 列數。bronze / document_text 一檔一列是小表；document_element 一元素一列較大，只 count。
if not settings["dry_run"]:
    t = table_name(settings, "bronze", PARSE_TABLE)
    df_latest = latest_per_doc(spark.table(t).select(*DOC_KEY, "category", "status", "parsed_at"))
    print(f"{t}（每檔最新一列）：")
    for r in df_latest.groupBy("category", "status").count().orderBy("category", "status").collect():
        print(f"  {r.category:<12} {r.status:<8} {r['count']}")
    for short in SILVER_TRANSFORMS:
        t = table_name(settings, "silver", short)
        print(f"{t}: rows={spark.table(t).count()}")
